In [1]:
import numpy as np
import torch
import os
import sys
import json
import jsonpickle as jpickle

current_dir = os.path.abspath('')
os.chdir(current_dir)
sys.path.append(os.path.join(current_dir,'code','BalancingControl'))

import two_stage_utils as tu
import inference_utils as iu
import inference as inf

torch threads 1


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on device cpu
torch threads 1


In [2]:
results_folder = "results"
model_comp_folder = os.path.join(results_folder, "model_comparison")
non_fit_fname = "non_fit_file.txt"
non_fit_file = os.path.join(model_comp_folder, non_fit_fname)

In [3]:
available_datasets_list = ["Chen_et_al_data", "FeherDaSilva_data/magic_carpet", "Kool_et_al_data/daw_paradigm"]

possible_models_list = ["BCC_2pars_planning", "BCC_4pars_planning_repetition_weight", 
                "BCC_4pars_planning_cached", "BCC_6pars_planning_repetition_weight_cached",
                "MFMB_4pars_mf_mb_Orig", "MFMB_6pars_mf_mb_Orig_prior"]

In [4]:
def load_measure(dataset, model, measure_name):

    measure_file = os.path.join(dataset, "results", "inference", model+"_inference", model+"_inference__"+measure_name+".json")

    with open(measure_file, 'r') as infile:
        pickled_measure = json.load(infile)
    measure = jpickle.decode(pickled_measure)

    return measure

In [5]:
measure_list = []
type_list = []
model_list = []
dataset_list = []

for dataset in available_datasets_list:

    for model in possible_models_list:

        # load WAIC
        WAIC = load_measure(dataset, model, "WAIC")
        measure_list.append(WAIC)
        type_list.append("WAIC")
        model_list.append(model)
        dataset_list.append(dataset)

        # load log likelihood
        ll = load_measure(dataset, model, "log_likelihood")
        measure_list.append(ll)
        type_list.append("log_likelihood")
        model_list.append(model)
        dataset_list.append(dataset)

In [6]:
WAIC_model_list = []
ll_model_list = []
type_list = []
model_list = []


for model in possible_models_list:
    WAIC_list = []
    ll_list = []

    for dataset in available_datasets_list:

        # load WAIC
        WAIC = load_measure(dataset, model, "WAIC")
        WAIC_list.append(WAIC)

        # load log likelihood
        ll = load_measure(dataset, model, "log_likelihood")
        ll_list.append(ll)

    WAIC_tensor = torch.cat(WAIC_list)
    ll_tensor = torch.cat(ll_list)

    WAIC_model_list.append(WAIC_tensor)
    ll_model_list.append(ll_tensor)
    model_list.append(model)

In [7]:
all_WAICs = torch.stack(WAIC_model_list, dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*all_WAICs)

p model mean according to measure tensor([0.1353, 0.0649, 0.0313, 0.1069, 0.2987, 0.3628])
best model: tensor(5) exceedance prob tensor(0.9500)
is significantly different from uniform? TtestResult(statistic=np.float64(181.66368222746834), pvalue=np.float64(0.0), df=np.int64(499))


In [8]:
all_WAICs = torch.stack(WAIC_model_list, dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*all_WAICs[...,:4])

p model mean according to measure tensor([0.3186, 0.2861, 0.0902, 0.3051])
best model: tensor(0) exceedance prob tensor(0.5520)
is significantly different from uniform? TtestResult(statistic=np.float64(66.65418486243715), pvalue=np.float64(6.711619036890671e-251), df=np.int64(499))


In [9]:
BCC6_MFMB6_WAICs = torch.stack([WAIC_model_list[3],WAIC_model_list[4]], dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*BCC6_MFMB6_WAICs)

p model mean according to measure tensor([0.2936, 0.7064])
best model: tensor(1) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(210.1493404287577), pvalue=np.float64(0.0), df=np.int64(499))


In [14]:
WAIC_model_list = []
ll_model_list = []
type_list = []
model_list = []


for model in possible_models_list:
    WAIC_list = []
    ll_list = []

    for dataset in available_datasets_list:

        non_fit_file = os.path.join(dataset, "results", "model_comparison", non_fit_fname)

        with open(non_fit_file, "r") as f:
            all_didnt_fit = json.load(f)

        # load WAIC
        WAIC = load_measure(dataset, model, "WAIC")

        did_fit = torch.ones_like(WAIC).bool()
        did_fit[all_didnt_fit] = False

        # print(did_fit.sum())

        WAIC_list.append(WAIC[did_fit])

        # load log likelihood
        ll = load_measure(dataset, model, "log_likelihood")
        ll_list.append(ll[did_fit])

    WAIC_tensor = torch.cat(WAIC_list)
    ll_tensor = torch.cat(ll_list)

    WAIC_model_list.append(WAIC_tensor)
    ll_model_list.append(ll_tensor)
    model_list.append(model)

In [15]:
all_WAICs = torch.stack(WAIC_model_list, dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*all_WAICs)

p model mean according to measure tensor([0.0624, 0.0801, 0.0328, 0.1320, 0.2493, 0.4433])
best model: tensor(5) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(229.31903263287666), pvalue=np.float64(0.0), df=np.int64(499))


In [23]:
iu.calculate_exceedance_prob(-0.5*all_WAICs[...,:4])

p model mean according to measure tensor([0.1695, 0.3516, 0.1025, 0.3764])
best model: tensor(3) exceedance prob tensor(0.7240)
is significantly different from uniform? TtestResult(statistic=np.float64(113.99706197544731), pvalue=np.float64(0.0), df=np.int64(499))


In [20]:
BCC6_MFMB6_WAICs = torch.stack([WAIC_model_list[3],WAIC_model_list[4]], dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*BCC6_MFMB6_WAICs)

p model mean according to measure tensor([0.3627, 0.6373])
best model: tensor(1) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(111.05155203362413), pvalue=np.float64(0.0), df=np.int64(499))
